# Experiment 7: percentile sweep for targeted training with a frozen FinBERT CNN

This notebook compares a frozen-FinBERT CNN baseline with targeted and matched random-control training across several hard-example percentiles. Hard examples are selected with the project's `utils.hard_samples_identify` margin utilities. Training selection uses out-of-fold predictions; validation and test subsets use the same percentile rule on baseline predictions. Results include overall and hard-subset accuracy/macro-F1 plus deltas relative to the baseline.

## Imports

In [3]:
import os
os.environ["HF_HOME"] = os.path.abspath("../.cache/huggingface")
os.environ["HF_HUB_CACHE"] = os.path.abspath("../.cache/huggingface/hub")
os.makedirs(os.environ["HF_HUB_CACHE"], exist_ok=True)
import pickle
import sys

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, f1_score
from sklearn.model_selection import StratifiedKFold
from tensorflow import keras
from tensorflow.keras import layers
from transformers import AutoModel, AutoTokenizer
from transformers import BertModel, BertTokenizerFast

sys.path.append("..")
from utils.preprocessing import LABELS, encode_labels
from utils.hard_samples_identify import get_hard_sample_indices

keras.utils.set_random_seed(42)


W1027 11:45:45.457000 5306 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


## Load FinBERT embeddings and tokenize

In [4]:
train_df = pd.read_csv("../data/train.csv")
val_df = pd.read_csv("../data/val.csv")
test_df = pd.read_csv("../data/test.csv")
y_train = encode_labels(train_df.label)
y_val = encode_labels(val_df.label)
y_test = encode_labels(test_df.label)

finbert_name = "yiyanghkust/finbert-pretrain"
tokenizer = BertTokenizerFast.from_pretrained(finbert_name)
finbert = BertModel.from_pretrained(finbert_name)
embedding_matrix = finbert.embeddings.word_embeddings.weight.detach().cpu().numpy()

X_train = tokenizer(train_df.sentence.tolist(), padding="max_length", truncation=True, max_length=64, return_tensors="np")["input_ids"]
X_val = tokenizer(val_df.sentence.tolist(), padding="max_length", truncation=True, max_length=64, return_tensors="np")["input_ids"]
X_test = tokenizer(test_df.sentence.tolist(), padding="max_length", truncation=True, max_length=64, return_tensors="np")["input_ids"]

del finbert
print(X_train.shape, X_val.shape, X_test.shape, embedding_matrix.shape)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 31606.58it/s]
[transformers] BertModel LOAD REPORT from: yiyanghkust/finbert-pretrain
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


(2901, 64) (967, 64) (968, 64) (30873, 768)


## Build the CNN

In [ ]:
def build_cnn():
    inputs = keras.Input(shape=(64,), dtype="int32")
    x = layers.Embedding(
        embedding_matrix.shape[0], 
        embedding_matrix.shape[1],
        embeddings_initializer=keras.initializers.Constant(embedding_matrix),
        trainable=False, mask_zero=False,
    )(inputs)
    x = layers.SpatialDropout1D(0.3)(x)
    x = layers.Conv1D(64, 3, activation="relu")(x)
    x = layers.GlobalMaxPooling1D()(x)
    x = layers.Dense(32, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(len(LABELS), activation="softmax")(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="categorical_crossentropy", metrics=["accuracy"])
    return model

class MacroF1Callback(keras.callbacks.Callback):
    def __init__(self, X, y):
        super().__init__(); self.X = X; self.y = y
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        predictions = self.model.predict(self.X, batch_size=32, verbose=0).argmax(axis=1)
        logs["val_macro_f1"] = f1_score(self.y, predictions, average="macro")
        print(f" - val_macro_f1: {logs['val_macro_f1']:.4f}")

def evaluate(model, X, y, name):
    probabilities = model.predict(X, batch_size=32, verbose=0)
    predictions = probabilities.argmax(axis=1)
    print(name)
    print("Accuracy:", accuracy_score(y, predictions))
    print("Macro F1:", f1_score(y, predictions, average="macro"))
    print(classification_report(y, predictions, target_names=LABELS))
    ConfusionMatrixDisplay.from_predictions(y, predictions, display_labels=LABELS)
    return probabilities, predictions


## Train the baseline

In [6]:
baseline_model = build_cnn()
baseline_model.fit(
    X_train, keras.utils.to_categorical(y_train, len(LABELS)),
    validation_data=(X_val, keras.utils.to_categorical(y_val, len(LABELS))),
    epochs=20, batch_size=32,
    callbacks=[MacroF1Callback(X_val, y_val), keras.callbacks.EarlyStopping(
        monitor="val_macro_f1", mode="max", patience=3, restore_best_weights=True
    )],
)


Epoch 1/20
90/91 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.5882 - loss: 0.9495 - val_macro_f1: 0.3184
91/91 ━━━━━━━━━━━━━━━━━━━━ 3s 27ms/step - accuracy: 0.5888 - loss: 0.9485 - val_accuracy: 0.6236 - val_loss: 0.8556 - val_macro_f1: 0.3184
Epoch 2/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.6412 - loss: 0.8309 - val_macro_f1: 0.4151
91/91 ━━━━━━━━━━━━━━━━━━━━ 3s 28ms/step - accuracy: 0.6412 - loss: 0.8309 - val_accuracy: 0.6784 - val_loss: 0.7745 - val_macro_f1: 0.4151
Epoch 3/20
90/91 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.6868 - loss: 0.7517 - val_macro_f1: 0.4368
91/91 ━━━━━━━━━━━━━━━━━━━━ 3s 35ms/step - accuracy: 0.6880 - loss: 0.7501 - val_accuracy: 0.6939 - val_loss: 0.7141 - val_macro_f1: 0.4368
Epoch 4/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.7218 - loss: 0.6512 - val_macro_f1: 0.5897
91/91 ━━━━━━━━━━━━━━━━━━━━ 3s 31ms/step - accuracy: 0.7218 - loss: 0.6512 - val_accuracy: 0.7332 - val_loss: 0.6519 - val_macro_f1: 0.5897
Epoch 5/20
9

## Mark hard examples

In [7]:
from utils.hard_samples_identify import get_hard_sample_indices, get_hard_sample_indices_by_margin

folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_probabilities = np.zeros((len(y_train), len(LABELS)))
for fold_train, fold_held_out in folds.split(X_train, y_train):
    fold_model = build_cnn()
    fold_model.fit(
        X_train[fold_train], keras.utils.to_categorical(y_train[fold_train], len(LABELS)),
        validation_split=0.1, epochs=20, batch_size=32, verbose=0,   # same recipe as baseline: early stopping
        callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)],
    )
    oof_probabilities[fold_held_out] = fold_model.predict(X_train[fold_held_out], batch_size=32, verbose=0)

val_probabilities = baseline_model.predict(X_val, batch_size=32, verbose=0)

# One cutoff per percentile, computed on train only, applied unchanged to val.
percentiles = [10, 20, 30, 40, 50]
hard_selection = {}
for percentile in percentiles:
    train_indices, train_cutoff = get_hard_sample_indices(oof_probabilities, percentile)
    val_indices = get_hard_sample_indices_by_margin(val_probabilities, train_cutoff)
    hard_selection[percentile] = {"train_indices": train_indices, "val_indices": val_indices,
                                  "train_cutoff": train_cutoff}
    print(f"P{percentile}: train={len(train_indices)}, val={len(val_indices)} (cutoff={train_cutoff:.4f})")


P10: train=290, val=44 (cutoff=0.2253)
P20: train=580, val=97 (cutoff=0.4194)
P30: train=870, val=144 (cutoff=0.5852)
P40: train=1160, val=188 (cutoff=0.7030)
P50: train=1451, val=229 (cutoff=0.7898)


In [10]:
from utils.hard_samples_identify import calculate_top_two_margins

# Save which sentences are hard, one is_hard_pXX column per percentile.
# Train: out-of-fold probabilities; val: baseline probabilities with the train cutoff.
os.makedirs("../results/hard_samples", exist_ok=True)
for split, df, probabilities in [("train", train_df, oof_probabilities), ("val", val_df, val_probabilities)]:
    out = df[["sentence", "label"]].copy()
    out["pred"] = [LABELS[i] for i in probabilities.argmax(axis=1)]
    out["margin"] = calculate_top_two_margins(probabilities)
    for percentile in percentiles:
        out[f"is_hard_p{percentile}"] = False
        out.loc[hard_selection[percentile][f"{split}_indices"], f"is_hard_p{percentile}"] = True
    out.to_csv(f"../results/hard_samples/exp7_{split}_hard.csv", index=False)
    print(split, {p: int(out[f"is_hard_p{p}"].sum()) for p in percentiles})

train {10: 290, 20: 580, 30: 870, 40: 1160, 50: 1451}
val {10: 44, 20: 97, 30: 144, 40: 188, 50: 229}


## Sweep targeted and random-control models


In [ ]:
def fine_tune(weighted_indices, seed, weight=1.5):
    keras.utils.set_random_seed(seed)
    model = keras.models.clone_model(baseline_model)
    model.set_weights(baseline_model.get_weights())
    model.compile(optimizer=keras.optimizers.Adam(1e-4), loss="categorical_crossentropy", metrics=["accuracy"])
    weights = np.ones(len(y_train))
    weights[weighted_indices] = weight
    model.fit(
        X_train, keras.utils.to_categorical(y_train, len(LABELS)), sample_weight=weights,
        validation_data=(X_val, keras.utils.to_categorical(y_val, len(LABELS))),
        epochs=10, batch_size=32, verbose=0,
        callbacks=[MacroF1Callback(X_val, y_val), keras.callbacks.EarlyStopping(
            monitor="val_macro_f1", mode="max", patience=1, restore_best_weights=True)],
    )
    return model

def random_indices(n, seed):
    return np.random.default_rng(seed).choice(len(y_train), size=n, replace=False)

## Compare percentile results


In [9]:
def score_model(model, X, y, hard_indices):
    probabilities = model.predict(X, batch_size=32, verbose=0)
    predictions = probabilities.argmax(axis=1)
    return {
        "accuracy": accuracy_score(y, predictions),
        "macro_f1": f1_score(y, predictions, average="macro"),
        "hard_accuracy": accuracy_score(y[hard_indices], predictions[hard_indices]),
        "hard_macro_f1": f1_score(y[hard_indices], predictions[hard_indices], average="macro"),
    }

SEEDS = [42, 43, 44, 45, 46]
rows = []
for percentile in percentiles:
    sel = hard_selection[percentile]
    base = score_model(baseline_model, X_val, y_val, sel["val_indices"])
    rows.append({"percentile": percentile, "strategy": "baseline", "seed": None, **base})
    for seed in SEEDS:
        for strategy, idx in [("targeted", sel["train_indices"]),
                              ("random_control", random_indices(len(sel["train_indices"]), seed))]:
            scores = score_model(fine_tune(idx, seed), X_val, y_val, sel["val_indices"])
            rows.append({"percentile": percentile, "strategy": strategy, "seed": seed, **scores})

results_df = pd.DataFrame(rows)
summary = (results_df.groupby(["percentile", "strategy"])[["accuracy", "macro_f1", "hard_accuracy", "hard_macro_f1"]]
           .agg(["mean", "std"]).round(4))
summary


 - val_macro_f1: 0.7361
 - val_macro_f1: 0.7472
 - val_macro_f1: 0.7441
 - val_macro_f1: 0.7368
 - val_macro_f1: 0.7494
 - val_macro_f1: 0.7401
 - val_macro_f1: 0.7414
 - val_macro_f1: 0.7360
 - val_macro_f1: 0.7387
 - val_macro_f1: 0.7352
 - val_macro_f1: 0.7392
 - val_macro_f1: 0.7393
 - val_macro_f1: 0.7461
 - val_macro_f1: 0.7497
 - val_macro_f1: 0.7375
 - val_macro_f1: 0.7370
 - val_macro_f1: 0.7369
 - val_macro_f1: 0.7445
 - val_macro_f1: 0.7673
 - val_macro_f1: 0.7444
 - val_macro_f1: 0.7396
 - val_macro_f1: 0.7625
 - val_macro_f1: 0.7418
 - val_macro_f1: 0.7472
 - val_macro_f1: 0.7455
 - val_macro_f1: 0.7472
 - val_macro_f1: 0.7476
 - val_macro_f1: 0.7463
 - val_macro_f1: 0.7385
 - val_macro_f1: 0.7472
 - val_macro_f1: 0.7358
 - val_macro_f1: 0.7361
 - val_macro_f1: 0.7504
 - val_macro_f1: 0.7393
 - val_macro_f1: 0.7414
 - val_macro_f1: 0.7348
 - val_macro_f1: 0.7387
 - val_macro_f1: 0.7363
 - val_macro_f1: 0.7392
 - val_macro_f1: 0.7410
 - val_macro_f1: 0.7463
 - val_macro_f1:

accuracy         macro_f1         hard_accuracy  \
                              mean     std     mean     std          mean   
percentile strategy                                                         
10         baseline         0.8046     NaN   0.7653     NaN        0.5455   
           random_control   0.7973  0.0062   0.7470  0.0102        0.5682   
           targeted         0.7990  0.0062   0.7505  0.0099        0.5773   
20         baseline         0.8046     NaN   0.7653     NaN        0.5979   
           random_control   0.7988  0.0072   0.7493  0.0122        0.6206   
           targeted         0.7983  0.0064   0.7498  0.0095        0.6247   
30         baseline         0.8046     NaN   0.7653     NaN        0.5903   
           random_control   0.7992  0.0060   0.7499  0.0096        0.5708   
           targeted         0.7986  0.0059   0.7501  0.0095        0.5694   
40         baseline         0.8046     NaN   0.7653     NaN        0.5851   
           random_control   0.7994  0.0054   0.7504  0.0100        0.5564   
           targeted         0.7990  0.0068   0.7503  0.0105        0.5521   
50         baseline         0.8046     NaN   0.7653     NaN        0.6026   
           random_control   0.7992  0.0062   0.7502  0.0103        0.5790   
           targeted         0.7983  0.0067   0.7493  0.0102        0.5764   

                                  hard_macro_f1          
                              std          mean     std  
percentile strategy                                      
10         baseline           NaN        0.5468     NaN  
           random_control  0.0278        0.3560  0.0426  
           targeted        0.0259        0.3705  0.0394  
20         baseline           NaN        0.5931     NaN  
           random_control  0.0295        0.4455  0.0346  
           targeted        0.0226        0.4511  0.0268  
30         baseline           NaN        0.5886     NaN  
           random_control  0.0352        0.4522  0.0681  
           targeted        0.0361        0.4472  0.0663  
40         baseline           NaN        0.5866     NaN  
           random_control  0.0295        0.4838  0.0520  
           targeted        0.0371        0.4728  0.0633  
50         baseline           NaN        0.5962     NaN  
           random_control  0.0270        0.5122  0.0456  
           targeted        0.0281        0.5100  0.0463

## Save percentile results


In [11]:
results_df.to_csv("../results/experiment_7_percentile_seeds.csv", index=False)
summary.to_csv("../results/experiment_7_percentile_summary.csv")


In [ ]:
# os.makedirs("../models", exist_ok=True)
# results_df.to_csv("../models/experiment_7_percentile_results.csv", index=False)
# with open("../models/experiment_7_finbert_tokenizer.pkl", "wb") as f:
#     pickle.dump(tokenizer, f)


In [30]:
pd.set_option("display.max_colwidth", None)   # show full text in every column

hard = pd.read_csv("../results/hard_samples/exp7_train_hard.csv")
hard[hard.is_hard_p50].sort_values("margin").head(30)[["sentence", "label", "pred", "margin"]]


,sentence,label,pred,margin
386,"The chain has been trying to cut its own costs , pressuring vendors such as fitness equipment manufacturer Precor to slash prices .",neutral,negative,0.001287
1464,The company said shareholders will be able to vote on the agreement at an EGM scheduled for later this month .,neutral,positive,0.002414
1579,"As previously announced , GeoSentric Oyj entered into financing agreements with its lead investor on June 30 , 2010 enabling the Company to receive financing up to the aggregate amount of 6M .",neutral,neutral,0.002808
2658,"Through its partnership with Finnish Paulig 's Danish subsidiary Oscar , Puljonki acquired a sales organization in the European market .",neutral,neutral,0.003727
1959,"The item included restructuring costs of EUR1 .6 m , while a year earlier they were EUR13 .1 m. Diluted EPS stood at EUR0 .3 versus a loss per share of EUR 0.1 .",positive,negative,0.003807
1082,"In the reporting period , EPS stood at EUR0 .07 versus EUR0 .11 .",negative,negative,0.004449
1812,"M-real Corporation Press release on 3 November at 10:35 am M-real Corporation was the second best company in the materials sector including e.g. forest industry companies in the Carbon Disclosure Leadership Index CDLI , Nordic Report 2009 .",positive,positive,0.004676
2837,"( ADP News ) - Sep 30 , 2008 - Finnish security and privacy software solutions developer Stonesoft Oyj said today that it won a USD 1.9 million ( EUR 1.3 m ) order to deliver its StoneGate network security products to an unnamed Russian te",positive,positive,0.006381
1091,Sanoma will continue to focus on investing in digital media and on strengthening its market positions .,neutral,positive,0.007741
1592,"Finnish Suominen Corporation that makes wipes , nonwovens , and flexible packaging , expects changes in the market situation to reduce sales of Suominen 's nonwovens and wet wipes from the previously estimated volumes .",negative,positive,0.007810


In [ ]:
# Weight sweep: is weight 1.5 just too mild? P20 only, 3 seeds (~3-4 min).
# weight 1.0 = continue training with no emphasis at all (extra control).
sel = hard_selection[20]
rows_w = []
for weight in [1.0, 2.0, 3.0]:
    for seed in [42, 43, 44]:
        for strategy, idx in [("targeted", sel["train_indices"]),
                              ("random_control", random_indices(len(sel["train_indices"]), seed))]:
            scores = score_model(fine_tune(idx, seed, weight), X_val, y_val, sel["val_indices"])
            rows_w.append({"weight": weight, "strategy": strategy, "seed": seed, **scores})

weight_df = pd.DataFrame(rows_w)
weight_df.to_csv("../results/experiment_7_weight_sweep_p20.csv", index=False)
weight_df.groupby(["weight", "strategy"])[["macro_f1", "hard_macro_f1"]].agg(["mean", "std"]).round(4)

In [31]:
def fine_tune(weighted_indices, seed, weight=1.5):
    keras.utils.set_random_seed(seed)
    model = keras.models.clone_model(baseline_model)
    model.set_weights(baseline_model.get_weights())
    model.compile(optimizer=keras.optimizers.Adam(1e-4), loss="categorical_crossentropy", metrics=["accuracy"])
    weights = np.ones(len(y_train))
    weights[weighted_indices] = weight
    model.fit(
        X_train, keras.utils.to_categorical(y_train, len(LABELS)), sample_weight=weights,
        validation_data=(X_val, keras.utils.to_categorical(y_val, len(LABELS))),
        epochs=10, batch_size=32, verbose=0,
        callbacks=[MacroF1Callback(X_val, y_val), keras.callbacks.EarlyStopping(
            monitor="val_macro_f1", mode="max", patience=1, restore_best_weights=True)],
    )
    return model

def random_indices(n, seed):
    return np.random.default_rng(seed).choice(len(y_train), size=n, replace=False)


In [32]:
# Weight sweep: is weight 1.5 just too mild? P20 only, 3 seeds (~3-4 min).
# weight 1.0 = continue training with no emphasis at all (extra control).
sel = hard_selection[20]
rows_w = []
for weight in [1.0, 2.0, 3.0]:
    for seed in [42, 43, 44]:
        for strategy, idx in [("targeted", sel["train_indices"]),
                              ("random_control", random_indices(len(sel["train_indices"]), seed))]:
            scores = score_model(fine_tune(idx, seed, weight), X_val, y_val, sel["val_indices"])
            rows_w.append({"weight": weight, "strategy": strategy, "seed": seed, **scores})

weight_df = pd.DataFrame(rows_w)
weight_df.to_csv("../results/experiment_7_weight_sweep_p20.csv", index=False)
weight_df.groupby(["weight", "strategy"])[["macro_f1", "hard_macro_f1"]].agg(["mean", "std"]).round(4)

 - val_macro_f1: 0.7368
 - val_macro_f1: 0.7494
 - val_macro_f1: 0.7390
 - val_macro_f1: 0.7368
 - val_macro_f1: 0.7494
 - val_macro_f1: 0.7390
 - val_macro_f1: 0.7387
 - val_macro_f1: 0.7355
 - val_macro_f1: 0.7387
 - val_macro_f1: 0.7355
 - val_macro_f1: 0.7384
 - val_macro_f1: 0.7393
 - val_macro_f1: 0.7461
 - val_macro_f1: 0.7471
 - val_macro_f1: 0.7461
 - val_macro_f1: 0.7384
 - val_macro_f1: 0.7393
 - val_macro_f1: 0.7461
 - val_macro_f1: 0.7471
 - val_macro_f1: 0.7461
 - val_macro_f1: 0.7482
 - val_macro_f1: 0.7452
 - val_macro_f1: 0.7385
 - val_macro_f1: 0.7507
 - val_macro_f1: 0.7374
 - val_macro_f1: 0.7422
 - val_macro_f1: 0.7326
 - val_macro_f1: 0.7387
 - val_macro_f1: 0.7387
 - val_macro_f1: 0.7438
 - val_macro_f1: 0.7434
 - val_macro_f1: 0.7392
 - val_macro_f1: 0.7407
 - val_macro_f1: 0.7456
 - val_macro_f1: 0.7455
 - val_macro_f1: 0.7376
 - val_macro_f1: 0.7358
 - val_macro_f1: 0.7490
 - val_macro_f1: 0.7452
 - val_macro_f1: 0.7385
 - val_macro_f1: 0.7563
 - val_macro_f1:

macro_f1         hard_macro_f1        
                          mean     std          mean     std
weight strategy                                             
1.0    random_control   0.7451  0.0056        0.4398  0.0198
       targeted         0.7451  0.0056        0.4398  0.0198
2.0    random_control   0.7440  0.0066        0.4268  0.0152
       targeted         0.7453  0.0030        0.4313  0.0132
3.0    random_control   0.7450  0.0099        0.4268  0.0152
       targeted         0.7465  0.0038        0.4313  0.0132